<a href="https://colab.research.google.com/github/DCC-Lab/STUAART/blob/master/Identify_location_One_cage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STUAART: Continuous Multi-Day Mass Location Identification

This Google Colab notebook is designed to identify location on scales from the mass data collected by the STUAART system.

Key Features:
- Multi-Day Continuity: Merges data from multiple daily CSV files into a single, linearized timeline.  
- Absolute Clock Alignment: Automatically calculates offsets based on the experiment's start time to view data in cumulative clock time format.  
- Ground Truth Validation: Overlays manual scale measurements directly onto the automated data for accuracy checks.  
- Signal Processing: Includes built-in convolution filters to smooth raw data and highlight underlying trends.  
- Flexible Windows: Supports precise "zooming" into specific hours or days using either relative time or clock-time strings.  

Instructions:
1. Ensure your raw STUAART CSV files are uploaded to your Google Drive.  
2. Update the file paths and experimental parameters in Step 2.  
3. Run all cells to generate individual scale plots and a combined smoothed mass trend.  

Note that you must run your raw data in *Remove buffer flush indicators in raw data.ipynb* before running this Colab.

# 1. Setup and Libraries

This cell imports the necessary Python libraries for data manipulation (pandas, numpy), visualization (matplotlib), and file system management.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
from itertools import groupby
from scipy.ndimage import uniform_filter1d
from scipy.interpolate import interp1d

# Step 2 : User configuration and Drive mounting

**The user must define these variables :**
1. Paths:
 - Change `project_directory` and `data_path `to point to your data in your Google Drive.

2. Visualisation:
 - `color_scales` : Defines the color of each scale.
 - `save_figure` : If True, figures are saved in a new folder *Figures*.
 - `figure_format` : A string that defines in which format you want your figures to be saved ("eps", "png", etc.)

3. Location identification:
 - `time_window` : Defines range of display. Use *None* to display the full duration, a tuple of numbers for **relative time** (e.g. (0,5)), or a tuple of strings for **absolute clock time** (e.g. ("8:00","12:00")).
 - `bins` : Defines the time (float) to bin the location data. Parameters such as `number_of_entries_per_bin_per_scale`, `average_time_spent_per_bin_per_scale`, `std_time_sent_per_bin_per_scale` and `relative_time_sent_per_bin_per_scale` are computed considering the time bin defined here. For example, 0.5 is 30 minutes.
  - `days` : A list of days to visualise (e.g. [1,2] or ["2026.04.01"]). Note that this variable must be a list even if only one day is selected.

4. Ground truth location (optional):
  - `ground_truth_file` : Defines the path where to find the location indicators per scale done manually by the experimenter. The CSV file should have the first column as the time stamps, and the following columns as an indicator, either 0 or 1, of the presence of the animal on the scale. This is done after, for example, video recording the STUAART to validate the performances of this notebook to identify the animal's location.

5. Ground truth mass (optional):
  - `manual_mass` : Add your manual scale measurements here as a list of tuples in the format (day, time, mass). The day can be either integers (e.g. 1) or strings (e.g. "2026.04.01"). The time can either be integers/floats (for relative time) or strings (for absolute clock time in the format "HH:MM")

`debug_mode` : If True, prints more informations for the user.

`save_figures` : If True, saves the figures in a folder called *Figures* in the figure format defined with `figure_format`.

`plot_figures` : If True, plots the figures in addition of printing the location computation parameter values.


After configuring these variables, the next cell will prompt you to mount your Google Drive to grant access to the files.

**Careful** : If `save_figure` is True but `plot_figure` is False, an error will occur.

In [ ]:
# Change these variables
project_directory = "/content/drive/My Drive/DCCM_Lab/labdata/vpineaunoel/stuaart"
data_path = project_directory + "/20260730-3MiceInSTUAARTs_5/Red/WithoutBufferFlushes-BaselineCorrectedAndOutlierRemoval-DBSCANwithManualCorrections/"
color_scales = ["b", "r", "g"]
save_figure = False
save_data = True
save_suffix = "Red5-Days4to6"
debug_mode = False
plot_figures = False
time_acquired_with_millis = False
bins = 0.166666666 # time to bin the data for location in float. For example, 0.5 is 30 minutes.
time_window = None # is either (number, number) for relative time, (str, str) for absolute clock time or None to display all data of the day
days = [4,5,6] # is either the day number (starting from 1) of the date in the format "YYYY.MM.DD". Must be a list.
sampling_rate = 80
figure_format = "pdf"
time_when_day_starts_end_ends = [7, 19]

# ground_truth_file = project_directory + "/20260730-3MiceInSTUAARTs_5/Pink/LocationGroundTruth/20260701-RecordBlueSTUAART-8943-Around16h28-DONE-GroundTruthLocationIndicatorPerSecond.csv"
ground_truth_file = None
# delay = -0.2822222222

# Add in the list the real mass measurements done during the experiment.
# Format : (day, time, mass)
# manual_mass = [
#     (2, "16:00", 24.7)
#     ]
manual_mass = None

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
os.chdir(project_directory)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


# 3. File Indexing and Date Extraction

This cell scans the provided data directory and sorts the CSV files. It automatically extracts the acquisition dates and assigns a relative day number (1, 2, 3...) to each file based on the filename prefix (e.g., YYYY.MM.DD-metadata.csv). This indexing allows the script to correctly map your selection in the days variable to the physical files on the drive.

In [ ]:
files = os.listdir(data_path)
files.sort()

dates_of_experiment = []
relative_days_of_experiment = []
i = 1
for file in files: # put in order the dates of the experiments based on the beginning of the CSV file names.
  if file.endswith(".csv"):
    index_first_dash = file.find("-")
    dates_of_experiment.append(file[:index_first_dash])
    relative_days_of_experiment.append(i)
    i += 1

if debug_mode:
  print("Dates of experiment : ", dates_of_experiment)
  print("Relative days of experiment : ", relative_days_of_experiment)

# 4. Time Mode Identification

This cell identifies the time mode according to the format of *time_window*.

- If *time_window* are strings (such as ("8:00","12:00")), *clock_time* is True.
- If *time_window* are integers/floats (such as (1.5, 2)), *clock_time* is False.
- If *time_window* is None, *clock_time* is None.

In [ ]:
if time_window is None: # no time window selected
  clock_time = False
elif all(isinstance(i, (int, float)) for i in time_window): # relative time
  clock_time = False
elif all(isinstance(i, (str)) for i in time_window): # absolute clock time
  clock_time = True

if debug_mode:
  print("Time window : ", time_window)
  print("Clock time : ", clock_time)

# 5. Linearizing the Time Window and Format Days

To allow for seamless visualization across multiple days, this cell "linearizes" your time selection. If you are using absolute clock time (e.g., from 15:00 on Day 1 to 10:00 on Day 2), the script converts these strings into a continuous float value. It achieves this by (1) converting "HH:MM" strings into decimal hours and (2) adding an offset of 24 hours for every day elapsed since the start of the experiment. For example, "02:00" on Day 2 becomes 26.0 hours ($2 + 24$). This creates a single, continuous timeline, making it possible to zoom into specific periods that span across midnight without breaking the data processing logic.

Additionally, the definition of days depends on the context and is fixed here. If both days and time_window are None, the entire time series across all days is considered. If days is None but time_window is provided, only the first day is considered.

Finally, the script calculates a start_shift based on the first file of the entire experiment (files[0]). This calculation ensures that the clock time remains consistent across all loaded files, allowing the script to correctly map all timestamps to the actual time of day.   

In [ ]:
def str_to_float_hour(time_str):
    """Convert str hour in the format 00:00 to a float value"""
    hours, minutes = map(int, time_str.split(':'))
    return hours + (minutes / 60.0)


def format_time_window_and_days(time_window, days, dates_of_experiment, relative_days_of_experiment, start_shift):
  if days is None and time_window is not None: # if no day is specified, but a time window is, the first day is considered
    days = [1]

  elif days is None and time_window is None: # if days and time_window are None, considers all data
    days = relative_days_of_experiment
    time_window = (0, 24)

  elif days is not None and time_window is None: # if time_window is None, all days defined in days are considered
    time_window = (0, 24)

  # Access first and last day of days
  start_day = days[0]
  end_day = days[-1]
  if all(isinstance(day, (int, float)) for day in days):
    formatted_start_day = days[0]
    formatted_end_day = days[-1]
  elif all(isinstance(day, str) for day in days):
    formatted_start_day = dates_of_experiment.index(days[0]) + 1
    formatted_end_day = dates_of_experiment.index(days[-1]) + 1
  else:
    raise ErrorValue("Days is not ints or strings. Verify the definition of days in cell 2.")

  if all(isinstance(time, (int, float)) for time in time_window):
    start_time_in_float = time_window[0] + start_shift
    end_time_in_float = time_window[1] + start_shift
  elif all(isinstance(time, str) for time in time_window):
    start_time_in_float = str_to_float_hour(time_window[0])
    end_time_in_float = str_to_float_hour(time_window[1])

  new_start_time = start_time_in_float + ((formatted_start_day-1) * 24)
  new_end_time = end_time_in_float + ((formatted_end_day-1) * 24)

  if debug_mode:
    print("Start time : ", start_time_in_float)
    print("End time : ", end_time_in_float)
    print("Formatted start day : ", formatted_start_day)
    print("Formatted end day : ", formatted_end_day)
    print("Time window after formatting : ", (new_start_time, new_end_time))
    print("Selected days : ", days)

  return days, (new_start_time, new_end_time)


def format_day_times(time_when_night_start_and_end, days, time_window):
  formatted_day_times = []
  for day in days:
    formatted_day_times.append((time_when_night_start_and_end[0] + ((day-1) * 24), time_when_night_start_and_end[1] + ((day-1) * 24)))
  return formatted_day_times

# compute start shift according to the time the experiment started on the first day
if time_acquired_with_millis:
  absolute_first_file = data_path + files[0]
  first_data = pd.read_csv(absolute_first_file)
  first_duration = first_data.iloc[-1, 0] / (1000 * 60 * 60)
  start_shift = (24.0 - first_duration) % 24
else:
  start_shift = 0

if debug_mode:
  print("Start shift : ", start_shift)

days, time_window = format_time_window_and_days(time_window=time_window, days=days, dates_of_experiment=dates_of_experiment, relative_days_of_experiment=relative_days_of_experiment, start_shift=start_shift)
day_times_per_day = format_day_times(time_when_night_start_and_end=time_when_day_starts_end_ends, days=days, time_window=time_window)

# 6. Data Loading and Global Alignment

This cell performs the core data aggregation and temporal alignment required for multi-day analysis.

1. Global Clock Alignment: The previously computed start_shift is used to align the time to the absolute clock time.   
2. Data Accumulation Loop: It iterates through your selected days, identifying the correct file paths whether you provided day numbers or specific dates. It then extracts the raw millisecond data, converts it into decimal hours, and applies the global shift to the dataset of each day.  
3. Merging and Filtering: All selected data points are merged into two primary arrays: *time_hour* and *mass_data*. Finally, the script uses a boolean mask to filter these arrays, retaining only the measurements that fall precisely within your defined *time_window*. This process enables the seamless visualization of a continuous timeline, even when the data spans across multiple files.

In [ ]:
all_time_hour = []
all_mass_data = []

if len(days) > 1:
  all_days = range(days[0], days[-1]+1, 1)
else:
  all_days = days

for d in all_days:
  if type(d) == str:
      actual_day_index = dates_of_experiment.index(d) # ex: 0, 1, 2...
      current_data_path = data_path + files[actual_day_index]
  else:
      actual_day_index = d - 1
      current_data_path = data_path + files[actual_day_index]

  if debug_mode:
    print("Data path : ", current_data_path)
  data = np.array(pd.read_csv(current_data_path))
  time = data[:,0]

  if time_acquired_with_millis:
    current_time_hour = time/(1000 * 60 * 60)
    current_time_hour = (start_shift + current_time_hour)
  else:
    current_time_hour = time

  current_mass_data = data[:, 1:]

  all_time_hour.append(current_time_hour)
  all_mass_data.append(current_mass_data)

print("All time hour : ", all_time_hour, len(all_time_hour), all_time_hour[0].shape)
# all data is contained in time_hour and mass_data after these two lines
time_hour = np.concatenate(all_time_hour)
mass_data = np.concatenate(all_mass_data, axis=0)

if time_acquired_with_millis is False:
  diffs = np.diff(time_hour)
  rollover_next_day = diffs < -12.0
  rollovers = np.concatenate(([False], rollover_next_day))
  days_passed = np.cumsum(rollovers)
  print("rollovers : ", rollovers, rollovers.shape, np.unique(rollovers))
  if 1.0 in all_days and np.any(rollovers):
    time_hour = time_hour + (days_passed * 24.0)
  elif not np.any(rollovers):
    time_hour = time_hour + ((d-1) * 24.0)
  else:
    indices_to_remove = np.where(days_passed == 0)[0]
    mass_data = np.delete(mass_data, indices_to_remove, axis=0)
    time_hour = np.delete(time_hour, indices_to_remove)
    time_hour = time_hour + ((d-1) * 24.0)

print(f"Beginning {time_hour[0]:.3f} and end {time_hour[-1]:.3f} value of time overall.")

mask = (time_hour >= time_window[0]) & (time_hour <= time_window[1])
time_in_window_hour = time_hour[mask]
mass_data_in_window = mass_data[mask, :]

# Initialize a mask of all True (Assume everything is night)
is_night = np.ones(len(time_in_window_hour), dtype=bool)
for day_start, day_end in day_times_per_day:
    is_day = (time_in_window_hour >= day_start) & (time_in_window_hour <= day_end)
    is_night[is_day] = False
night_indicator_list = is_night.astype(int)

if debug_mode:
    print("Time window input : ", time_window)
    if len(time_hour) > 0:
      print(f"Time window in absolute clock time : Start {time_in_window_hour[0]:.3f} and End {time_in_window_hour[-1]:.3f}")
    else:
      print("No data found in mask")

    print("Time_hour : ", time_hour, time_hour.shape)
    print(f"Time_hour_in_window : {time_in_window_hour}{time_in_window_hour.shape}")
    print("Mass_data : ", mass_data_in_window, mass_data_in_window.shape)
    print("Night indicator list : ", np.unique(night_indicator_list), night_indicator_list.shape)

All time hour :  [array([2.39997944e+01, 7.55555556e-05, 7.88888889e-05, ...,
       2.39997697e+01, 2.39997733e+01, 2.39997767e+01]), array([2.39997803e+01, 6.13888889e-05, 6.50000000e-05, ...,
       2.39997497e+01, 2.39997533e+01, 2.39997567e+01]), array([2.39997603e+01, 4.13888889e-05, 4.50000000e-05, ...,
       8.73214778e+00, 8.73215111e+00, 8.73215472e+00])] 3 (6698191,)
rollovers :  [False  True False ... False False False] (15849615,) [False  True]
Beginning 120.000 and end 128.732 value of time overall.


# 7. Processing and Filtering Manual Measurements (Ground Truth)

This cell handles the manual scale measurements (ground truth) provided by the user, ensuring they are correctly positioned on the continuous multi-day timeline.

1. Time Linearization (*format_manual_mass()*): Just as the raw data was shifted to create a continuous timeline, this function converts the *manual_mass* tuples into linearized values. It identifies the correct day of the experiment and adds a 24-hour offset for each elapsed day. This ensures that a manual weighing at 10:00 AM on Day 2 is accurately mapped to Hour 34.0 on the graph.  
2. Window Filtering (*filter_manual_measurements()*): This function sifts through the linearized manual measurements and retains only those that fall within your *time_window*.  

Output: The resulting *real_mass* variable contains the specific data points that will be overlaid on the final plot for visual validation of the STUAART accuracy to track the mass.

In [ ]:
def filter_manual_measurements(measurements, current_days, dates_of_experiment, current_window, is_clock_mode):
  filtered_times = []
  filtered_masses = []

  for m_time, m_mass in measurements:
    # convert time in float
    if isinstance(m_time, str):
      t_float = str_to_float_hour(m_time) # function defined in that last cell
    else:
      t_float = m_time # relative time is kept as is

    # check time
    if current_window is not None:
      w_start = str_to_float_hour(current_window[0]) if isinstance(current_window[0], str) else current_window[0]
      w_end = str_to_float_hour(current_window[1]) if isinstance(current_window[1], str) else current_window[1]

    if not (w_start <= t_float <= w_end):
      continue

    filtered_times.append(t_float)
    filtered_masses.append(m_mass)

  return filtered_times, filtered_masses


def format_manual_mass(manual_mass, dates_of_experiment, relative_days_of_experiment, start_shift):
  if manual_mass is None:
    return None
  else:
    new_manual_mass = []
    for m_day, m_time, m_mass in manual_mass:
      if isinstance(m_day, (int, float)):
        new_day = relative_days_of_experiment[m_day-1]
      elif isinstance(m_day, str):
        new_day = dates_of_experiment.index(m_day) + 1

      if isinstance(m_time, str):
        time_in_float = str_to_float_hour(m_time)
      else:
        time_in_float = m_time + start_shift

      new_time = time_in_float + ((new_day-1) * 24)
      new_manual_mass.append((new_time, m_mass))

  return new_manual_mass


if manual_mass is not None:
  if debug_mode:
    print("Manual mass as input : ", manual_mass)
    print("Type of manual mass : ", type(manual_mass))
    print("Current days : ", days)
    print("Current time window : ", time_window, type(time_window))

  new_manual_mass = format_manual_mass(manual_mass=manual_mass, dates_of_experiment=dates_of_experiment, relative_days_of_experiment=relative_days_of_experiment, start_shift=start_shift)
  real_mass = filter_manual_measurements(measurements=new_manual_mass, current_days=days, dates_of_experiment=dates_of_experiment, current_window=time_window, is_clock_mode=clock_time)

  if debug_mode:
    print("Real mass after formatting : ", new_manual_mass)
    print("Real mass after filtering with time_window : ", real_mass)

else:
  real_mass = None

# 8. Total Mass Computation

This cell calculates the total mass of the entire STUAART. `sum_data_over_time()` sums the values from all scales at each timestamp to provide a single total cage mass measurement.  


In [ ]:
def sum_data_over_time(data_per_scale):
  """
  Sum data per time point to obtain the mass measurement over time of the whole cage system (and not only per scale).
  """
  summed_data = np.nansum(data_per_scale, axis=1)
  return summed_data

summed_data_in_window = sum_data_over_time(data_per_scale=mass_data_in_window)

if debug_mode:
  print("Shape of mass data before summing mass over every timestamp: ", mass_data_in_window.shape)
  print("Shape of summed data : ", summed_data_in_window.shape)
  print("Shape of time : ", time_in_window_hour.shape)

# 9. Spatial Tracking and Mass Data Signal Processing

This cell contains the core pipeline for processing raw mass data from the individual cage scales to track the mouse's location and quantify its presence behaviors over time.

1. `when_mouse_is_in(...)` : Detects the exact entry timestamp of the mouse on the first day by identifying when the total mass signal permanently leaves the $0\text{ g}$ baseline (ignoring initial calibration plateaus).

2. `convolution_filter_with_padding_edge_average_per_scale(...)` : Smooths individual scale signals and eliminates random outliers using a moving average convolution filter.

3. `compute_location_on_scale(...)` : Binarizes the smoothed scale data to determine if a mouse is present on a given scale (mass $> \pm 2\text{ g}$). For customizable time windows (`bins`), it calculates:
     - **Number of entries** (bouts) per scale.
     - **Average & standard deviation of time spent** per entry.
     - **Relative time spent** on each scale.

In [ ]:
def when_mouse_is_in(time, data, sampling_rate):
  """
  Identifies the moment when the mouse is in the cage.
  Data acquisition starts with a plateau at 0g. When the mouse is in, the mean mass goes up.
  Creates two class variables with the time and the mass data when the mouse is in.
  """
  length = sampling_rate
  for i in range(data.shape[0]):
    mean = np.nanmean(data[i: i+length])
    if np.isclose(mean, 0, atol=3): # + or - one gram is still considered at 0g.
      i += length
    else:
      break
  time_when_mouse_is_in = time[i+length]

  return time_when_mouse_is_in


def convolution_filter_with_padding_edge_average_per_scale(data_per_scale, length: int = 10, iteration: int = 1):
  if iteration <= 0:
    raise ValueError("Number of iteration cannot be under 1")

  current_data = data_per_scale.copy()

  for _ in range(iteration):
    valid_mask = ~np.isnan(current_data)
    data_filled = np.nan_to_num(current_data, nan=0.0)

    # Vectorized across all scales at once using axis=0
    moving_sum = (uniform_filter1d(data_filled, size=length, axis=0, mode='nearest') * length)
    moving_count = (uniform_filter1d(valid_mask.astype(float), size=length, axis=0, mode='nearest') * length)

    with np.errstate(divide='ignore', invalid='ignore'):
      current_data = moving_sum / moving_count
      current_data[moving_count == 0] = np.nan

    return current_data



def compute_location_on_scale(time, data_per_scale, summed_data, sampling_rate, bins: float = 0.5, first_day: bool = False):
  """
  Time series of mass per individual scale are used to identify where the mouse is over time.
  After a simple average convolution, the mouse is identified as present on a scale if
  mass is outside noise threshold (-5 to 5g).
  """
  convolved_data = convolution_filter_with_padding_edge_average_per_scale(data_per_scale=data_per_scale, length=10)
  presence_indicator_per_scale = np.ones(shape=convolved_data.shape)
  number_of_scales = convolved_data.shape[1]

  if first_day:
    time_when_mouse_is_in = when_mouse_is_in(time=time, data=summed_data, sampling_rate=sampling_rate)
    # Safe float match using minimum distance
    index_when_mouse_is_in = np.argmin(np.abs(time - time_when_mouse_is_in))
    presence_indicator_per_scale[:index_when_mouse_is_in, :] = 0

  for n in range(number_of_scales):
    index_data_around_zero = np.where((convolved_data[:, n] > -5) & (convolved_data[:, n] < 5))[0]
    presence_indicator_per_scale[index_data_around_zero, n] = 0

  # Compute metrics per scale
  number_of_entries_per_bin_per_scale = []
  average_time_spent_per_bin_on_scale = []
  std_time_spent_per_bin_on_scale = []
  relative_time_spent_per_bin_per_scale = []

  dt = time[1] - time[0]
  bin_edges = np.arange(time[0], time[-1] + bins, bins)

  for n in range(number_of_scales):
    number_of_entries = []
    average_time_spent = []
    std_time_spent = []
    relative_time = []

    for start_t, end_t in zip(bin_edges[:-1], bin_edges[1:]):
      mask = (time >= start_t) & (time < end_t)
      segment = presence_indicator_per_scale[mask, n]

      # Guard against empty time intervals
      if segment.size == 0:
        number_of_entries.append(0)
        average_time_spent.append(0.0)
        std_time_spent.append(0.0)
        relative_time.append(0.0)
        continue

      # Extract consecutive bout lengths where presence == 1
      bout_sample_lengths = [len(list(group)) for key, group in groupby(segment) if key == 1]

      if len(bout_sample_lengths) > 0:
        lengths_in_seconds = np.array(bout_sample_lengths) * dt
        num_entries = len(bout_sample_lengths)
        mean_time = np.mean(lengths_in_seconds)
        std_time = np.std(lengths_in_seconds)
        rel_time = np.sum(lengths_in_seconds) / bins
      else:
        num_entries = 0
        mean_time = 0.0
        std_time = 0.0
        rel_time = 0.0

      number_of_entries.append(num_entries)
      average_time_spent.append(mean_time)
      std_time_spent.append(std_time)
      relative_time.append(rel_time)

    number_of_entries_per_bin_per_scale.append(np.array(number_of_entries))
    average_time_spent_per_bin_on_scale.append(np.array(average_time_spent))
    std_time_spent_per_bin_on_scale.append(np.array(std_time_spent))
    relative_time_spent_per_bin_per_scale.append(np.array(relative_time))

  return (
    convolved_data,
    presence_indicator_per_scale,
    np.array(number_of_entries_per_bin_per_scale),
    np.array(average_time_spent_per_bin_on_scale),
    np.array(std_time_spent_per_bin_on_scale),
    np.array(relative_time_spent_per_bin_per_scale)
  )

if relative_days_of_experiment[0] == days[0]:
  first_day = True
else:
  first_day = False

convolved_data_per_scale, presence_indicator_per_scale, number_of_entries_per_bin_per_scale, average_time_spent_per_bin_per_scale, std_time_sent_per_bin_per_scale, relative_time_sent_per_bin_per_scale = compute_location_on_scale(time=time_in_window_hour, data_per_scale=mass_data_in_window, summed_data=summed_data_in_window, sampling_rate=sampling_rate, first_day=first_day, bins=bins)

if debug_mode:
  print(f"Convolved data per scale : \n {convolved_data_per_scale}, {convolved_data_per_scale.shape}")
  print(f"Presence indicator per scale : \n {presence_indicator_per_scale}, {presence_indicator_per_scale.shape}, {np.unique(presence_indicator_per_scale)}")
  print(f"Number of entries per bin per scale : \n {number_of_entries_per_bin_per_scale}, {number_of_entries_per_bin_per_scale.shape}")
  print(f"Average time spent per bin per scale : \n {average_time_spent_per_bin_per_scale} +- {std_time_sent_per_bin_per_scale}")
  print(f"Relative time spent per bin per scale : \n {relative_time_sent_per_bin_per_scale}, {relative_time_sent_per_bin_per_scale.shape}")

# 10. Format ground truth labels (optional)

This section automates the temporal alignment between high-resolution experimental data and reference ground-truth data. If `ground_truth_data` is `None`, this section is skipped.

`find_optimal_time_delay()` : Finds the time delay by scanning a range of candidate delays and maximizing the overlap of presence states (`1`s) between the automated detections and the ground truth detections.

In [ ]:
def find_optimal_time_delay(high_res_array, high_res_time, low_res_array, low_res_time, search_range_hours=(-1.0, 1.0), step_seconds=1.0,):
  """
  Finds the true time delay by scanning a range of candidate delays
  and maximizing the overlap of presence (1s). Supports both 1D and 2D arrays.
  """
  dt_hours = step_seconds / 3600.0
  delays = np.arange(search_range_hours[0], search_range_hours[1] + dt_hours, dt_hours)

  f_base = interp1d(low_res_time,low_res_array,kind='nearest',bounds_error=False,fill_value=0,)
  best_delay = 0
  max_overlap = -1

  for tau in delays:
    y_candidate = f_base(high_res_time - tau)
    overlap = np.sum((high_res_array == 1) & (y_candidate == 1))

    if overlap > max_overlap:
      max_overlap = overlap
      best_delay = tau

  return best_delay


if ground_truth_file is not None:
  df_ground_truth_data = pd.read_csv(ground_truth_file, index_col=0)
  time_ground_truth = np.array(df_ground_truth_data.iloc[:, 0])
  cumulative_time_ground_truth = time_ground_truth+((days[0]-1)*24)
  indicators_per_scale_ground_truth = df_ground_truth_data.iloc[:, 1:]
  indicators_per_scale_ground_truth = np.array(indicators_per_scale_ground_truth).T

  if delay is None:
    all_dts = []
    for i in range(indicators_per_scale_ground_truth.shape[0]):
      dt = find_optimal_time_delay(
          high_res_array=presence_indicator_per_scale[:,i],
          high_res_time=time_in_window_hour,
          low_res_array=indicators_per_scale_ground_truth[i],
          low_res_time=cumulative_time_ground_truth
          )
      all_dts.append(dt)
    dt = np.mean(all_dts)
  else:
    dt = delay

  if debug_mode:
    print("Time delay for optimal overlap : ", dt)
  ground_truth_data = (cumulative_time_ground_truth + dt, indicators_per_scale_ground_truth)
else:
  ground_truth_data = None

if debug_mode:
  if ground_truth_file is None:
    pass
  else:
    print("Ground truth data time : ", ground_truth_data[0].shape)
    print("Ground truth data indicators per scale : ", ground_truth_data[1], ground_truth_data[1].shape, np.unique(ground_truth_data[1]))

# 11. Data Visualisation

This cell displays the mass time series from individual scales if `plot_figure` is True. Each scale is plotted on its own axis, allowing the inspection of data from each scales independently. If *manual_mass* is not empty, those measurements are overlayed on the mass time series with a yellow star for immediate visual comparison. The time is displayed as a cumulative time of day. If `save_figure` is True, the figure is saved in a folder named *Figures*.

In [ ]:
def plot_results_of_location_analysis(time, data_per_scale, colors, figure_format, number_of_entries_per_bin_per_scale, average_time_spent_per_bin_per_scale, std_time_sent_per_bin_per_scale, relative_time_sent_per_bin_per_scale, is_saved, bins:float=0.5):
  number_of_scales = data_per_scale.shape[1]
  num_bins = number_of_entries_per_bin_per_scale.shape[1]

  fig, axs = plt.subplots(ncols=1, nrows=3, figsize=(20,30))
  x = np.arange(num_bins)
  total_group_width = 0.6
  bar_width = total_group_width / number_of_scales
  for n in range(number_of_scales):
    offset = (n - (number_of_scales - 1) / 2) * bar_width
    current_x = x + offset
    axs[0].bar(current_x, number_of_entries_per_bin_per_scale[n], color=colors[n], width=bar_width, alpha=0.75, label="Scale "+str(n+1))
    axs[1].bar(current_x, average_time_spent_per_bin_per_scale[n], yerr=std_time_sent_per_bin_per_scale[n], error_kw={'ecolor':colors[n] ,'elinewidth': 3,'linestyle': '--','capthick': 5}, color=colors[n], width=bar_width, alpha=0.75)
    axs[2].bar(current_x, relative_time_sent_per_bin_per_scale[n], color=colors[n], width=bar_width, alpha=0.75)

  plt.xlabel("Time bins [h]", fontsize=100)
  axs[0].legend(fontsize=100)
  axs[0].set_ylabel("Number of entries", fontsize=100)
  axs[1].set_ylabel("Average time \n spent per \n presence bout [h]", fontsize=100)
  axs[2].set_ylabel("Relative time [h/h]", fontsize=100)

  real_time_labels = (bins) + bins * np.arange(num_bins)

  for ax in axs:
    ax.set_ylim(bottom=0)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{val:.2f}" for val in real_time_labels], fontsize=60)
    ax.tick_params(axis='y', labelsize=60)
  fig.tight_layout()

  if is_saved:
    path = Path(data_path)
    figure_path = path.parent / "Figures"
    figure_path.mkdir(parents=True, exist_ok=True)
    file_stem = path.stem
    if time_window is None:
      suffix = "-Entries_Averagetimeperbout_Relativetime." + figure_format
    else:
      suffix = f"-Entries_Averagetimeperbout_Relativetime-Between{time_window[0]:.3f}and{time_window[1]:.3f}hours-Raw." + figure_format

    save_path = figure_path / (file_stem + suffix)
    fig.tight_layout()
    if figure_format == "png":
      plt.savefig(save_path, format=figure_format, dpi=600, transparent=True, facecolor='white', edgecolor='none')
    else:
      plt.savefig(save_path, format=figure_format, transparent=True, facecolor='white', edgecolor='none')
  plt.show()


def plot_raw_mass_time_series_with_location_indicator(time, data, presence_indicator_per_scale, is_saved, colors, real_mass, clock_time, time_window, figure_format, night_indicator_list, ground_truth=None):
  fig, axs = plt.subplots(data.shape[1], 1, figsize=(30,15))

  for i in range(data.shape[1]):
    axs[i].plot(time, data[:,i], color=colors[i], label = "Mass on scale " + str(i+1), zorder=4)
    axs[i].fill_between(time, np.nanmax(data[:,i]), where=presence_indicator_per_scale[:,i] == 1, color=colors[i], alpha=0.5, zorder=3)
    if real_mass is not None:
      axs[i].scatter(real_mass[0], real_mass[1], marker="*", edgecolors="k", color="m", label="Real mass", s=1000, zorder=5)

    axs[i].set_xlabel("Cumulative time of day [h]", fontsize=50)
    axs[i].set_ylabel("Mass [g]", fontsize=50)
    axs[i].tick_params(axis='both', labelsize=35)
    axs[i].set_ylim(-5, 50)

    # nights have a lightgray background
    axs[i].fill_between(time, 0, 1, where=is_night, color='lightgray', alpha=1, transform=axs[i].get_xaxis_transform(), label="Night", zorder=1)

    if ground_truth is not None:
      axs[i].fill_between(ground_truth[0], np.nanmax(data[:,i]), where=ground_truth[1][i], color='lightgray', alpha=0.4, transform=axs[i].get_xaxis_transform(), label="Ground truth location labels", zorder=2)

    axs[i].legend(loc="upper right", fontsize=35)

  if is_saved:
    path = Path(data_path)
    figure_path = path.parent / "Figures"
    figure_path.mkdir(parents=True, exist_ok=True)
    file_stem = path.stem
    if time_window is None:
      suffix = "-Raw." + figure_format
    else:
      suffix = f"-RelativeTimeWithLocationIndicator-Between{time_window[0]:.3f}and{time_window[1]:.3f}hours-Raw." + figure_format

    save_path = figure_path / (file_stem + suffix)
    fig.tight_layout()
    plt.savefig(save_path, format=figure_format, dpi=600, transparent=True, facecolor='white', edgecolor='none')
  plt.show()


if plot_figures:

  plot_raw_mass_time_series_with_location_indicator(
      time=time_in_window_hour,
      data=convolved_data_per_scale,
      presence_indicator_per_scale=presence_indicator_per_scale,
      is_saved=save_figure, colors=color_scales,
      real_mass=real_mass, clock_time=clock_time,
      time_window=time_window,
      figure_format=figure_format,
      night_indicator_list=night_indicator_list,
      ground_truth=ground_truth_data)

  plot_results_of_location_analysis(
      time=time_in_window_hour,
      data_per_scale=mass_data_in_window,
      colors=color_scales,
      number_of_entries_per_bin_per_scale=number_of_entries_per_bin_per_scale,
      average_time_spent_per_bin_per_scale=average_time_spent_per_bin_per_scale,
      std_time_sent_per_bin_per_scale=std_time_sent_per_bin_per_scale,
      relative_time_sent_per_bin_per_scale=relative_time_sent_per_bin_per_scale,
      is_saved=save_figure,
      figure_format=figure_format,
      bins=bins)

# 12. Save presence indicators per scale

If `save_data` is True, the mass time series in the selected `time_window` at the selected `days` is saved as well as the presence indicators.

In [ ]:
if save_data:
  data ={"Time in time window [h]":time_in_window_hour, "Night indicator":night_indicator_list}
  for i in range(presence_indicator_per_scale.shape[1]):
    data[f"Convolved mass in time window - Scale {i+1}"] = mass_data_in_window[:,i]
    data[f"Presence indicator in timw window - Scale {i+1}"] = presence_indicator_per_scale[:,i]
  df = pd.DataFrame(data)
  path = Path(data_path)
  saved_path = path.parent / "Results of Location Analysis"
  saved_path.mkdir(parents=True, exist_ok=True)
  file_stem = path.stem
  suffix = save_suffix + "-ConvolvedMass_PresenceIndicatorPerScale_NightIndicator.csv"
  save_path = saved_path / (file_stem + suffix)
  df.to_csv(save_path)